Импортируем необходимые библиотеки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import phik
from phik.report import plot_correlation_matrix

from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from lightgbm import LGBMRegressor

pd.set_option('display.max_columns', None)

### Изучение общей информации о данных

Прочитаем файл с данными и изучим общую информацию о данных

In [ ]:
dt = pd.read_csv('./notebooks/data/data_final.csv')
display(dt.info())
display(dt.head())

Видим, что не у всех данных корректный тип данных, поэтому приведем столбцы датафрейма к корректным типам. first_seen нужно сделать датой, количественные показатели числом, а бинарные сделать типом bool.

In [ ]:
dt['first_seen'] = pd.to_datetime(dt['first_seen'],format='ISO8601')
dt[['rooms','total_lifts','views_total']] = dt[['rooms','total_lifts','views_total']].astype('Int64')
dt[['event_closed','mortgage_allowed','nearest_metro_walk','is_studio','is_apartments','is_new_building',
    'phone_protected','had_discount','is_first_floor','is_last_floor','has_lift',
    'demolished_in_renovation','is_penthouse','seller_is_owner']] = dt[['event_closed','mortgage_allowed',
                                                                                         'nearest_metro_walk','is_studio','is_apartments','is_new_building',
                                                                                         'phone_protected','had_discount','is_first_floor','is_last_floor','has_lift',
                                                                                         'demolished_in_renovation','is_penthouse','seller_is_owner']].astype('bool')

Посмотрим на данные еще раз, чтобы убедиться, что типы данных теперь корректные и соответствуют по смыслу.

In [ ]:
dt.info()

Посмотрим на количество пропусков

In [ ]:
dt.isna().sum().sort_values(ascending = False)

Видим, что пропусков нет нигде, кроме трех полей: в views_today, photos_count и views_total.

In [ ]:
(dt == 'unknown').sum().sort_values(ascending = False)

Посмотрим, сколько unknown есть в данных, их значительно больше, чем пропусков, нас особенно беспокоит 95% по district

In [ ]:
dt.describe()

Посмотрели какие аномалии есть в данных. Нас особенно смущает, что 
- rooms = 0 (min), так как не бывает квартир, где нет комнат
- floor = -2 (min) - отрицательное значение этажа
- ceiling_height = 0 (min), 3250 (max) - не бывает потолков, где высота 0
- views_total = 536340 (max) - здесь нас смутило слишком большое кол-во просмотров
- dist_to_center = 2811.959345 (max) - подозрительно большое расстояние для городской недвижимости
- building_age = -3 (min), 934 (max), нереалистичный возраст здания
- floor_ratio = -0.4 (min), не имеет смысла отрицательное значение
- total_lifts = 43 (max) - так не бывает 

Посмотрим какие уникальные значения есть в каждом столбце, чтобы определить что нужно поправить, отфильтровать

In [ ]:
for col in dt.select_dtypes(include = 'object').columns:
    print()
    print(col, dt[col].nunique())
    print(dt[col].value_counts().head(10))

# investment,dzhsk,свободная продажа -> alternative
# отфильтровать регионы - Москва и МО
# проверить district
# привести bathrooms?
# привести balcony?
# window_view -> на сколько сторон окна: yardAndStreet = 2, yard = 1, street = 1
# привести building_type

Посмотрим сколько дней объявление было показано

In [ ]:
plt.hist(dt['days_on_market'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)

plt.xlabel('Число дней на рынке')
plt.ylabel('Частота')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

На графике видим, что большинство квартир были на рынке небольшое количество дней и быстро продавались.

In [ ]:
dt = dt[dt['region'].isin(['Москва','Московская область'])]
dt.loc[dt['views_total']>100000, 'views_total'] = np.nan
dt = dt[dt['ceiling_height'].between(2, 8)]
dt = dt[dt['building_age']<300]
dt['price_range_pct'] = dt['price_range_pct'].clip(upper=dt['price_range_pct'].quantile(0.99))
dom_low, dom_high = dt['days_on_market'].quantile([0.01, 0.99])
dt = dt[dt['days_on_market'].between(dom_low, dom_high)]

Очистим данные, удалим ненужные и отфильтруем аномальные значения.
- удалим views_today и photos_count - счетные признаки, более 80% пропусков, малая содержательная нагрузка + views_total
- отфильтруем только объявления в Москве и МО. Это должно также убрать выбросы по dist_to_center, lat, lon
- что касается rooms = 0 (min) - вероятно, свободная планировка, оставим пока. floor = -2 (min) - сомнительно, но окэй :)
- views_total = 536340 - такой выброс всего один, следующее максимальное значение 58733, скорее всего ошибка. Присвоим ему NaN.
- ceiling_height - возьмем от 2 метров до 8 метров (пентхаус), остальные скорее всего ошибка
- building_age = -3 (min), 934 (max) - верхнее значение точно ошибка, это Подольск, ему еще нет столько лет. А нижние (отрицательные) значения - если это в строящихся домах, то почему маркер is_ready у всех true? Надо разобраться.
- days_on_market - срезаем хвосты по 1 и 99 перцентилю (снизу мгновенные репосты, сверху "продается 10+ лет")

Посмотрим зависимости для цены за кв.м. и переменных с наибольшей корреляцией, а также зависимости для срока продажи и переменных с наибольшей корреляцией

In [ ]:
pairs = [
    ('n_metro', 'price_per_m2'),
    ('total_floors', 'price_per_m2'),
    ('floor', 'price_per_m2')
]

fig, axes = plt.subplots(1, len(pairs), figsize=(16, 5))

for ax, (x_col, y_col) in zip(axes, pairs):
    sns.scatterplot(data=dt, x=x_col, y=y_col, alpha=0.3, s=10, ax=ax)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f'{y_col} vs {x_col}')

plt.suptitle('Зависимость цены за кв.м. от ключевых факторов', y=1.02)
plt.tight_layout()
plt.show()

Видим, что цена за квадратный метр растёт с увеличением числа станций метро рядом и снижается в многоэтажных домах. Наиболее дорогие квартиры расположены на средних и низких этажах.



In [ ]:
pairs = [
    ('living_area', 'days_on_market'),
    ('views_total', 'days_on_market'),
    ('price_points', 'days_on_market')
]

fig, axes = plt.subplots(1, len(pairs), figsize=(16, 5))

for ax, (x_col, y_col) in zip(axes, pairs):
    sns.scatterplot(data=dt, x=x_col, y=y_col, alpha=0.3, s=10, ax=ax)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f'{y_col} vs {x_col}')

plt.suptitle('Зависимость срока продажи от ключевых факторов', y=1.02)
plt.tight_layout()
plt.show()

Посмотрим на распределение цен за кв. метр

Срок продажи увеличивается для квартир с большой жилой площадью и низкой ценой за квадратный метр. А также видим, что чем больше просмотров, тем быстрее продается квартира.



In [ ]:
cols_to_plot = ['district', 'flat_type', 'building_type']

for col in cols_to_plot:
    dt_clean = dt[dt[col].ne('unknown') & dt[col].notna()]
    top_cats = dt_clean[col].value_counts().head(10).index
    dt_top = dt_clean[dt_clean[col].isin(top_cats)]
    
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=dt_top, x=col, y='price_per_m2', order=top_cats)
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Цена за кв.м. по топ-10 {col} (по кол-ву строк)')
    plt.ticklabel_format(style='plain', axis='y')
    plt.tight_layout()
    plt.show()


Видим, что самые высокие средние цены за кв. метр в районах Раменки и Пресненский, а также в студиях и домах сталинской постройки.

### Распределение таргета (days_on_market) по категориальным признакам

In [ ]:
def target_by_category(frame, col, target='days_on_market', min_count=200):
    g = frame.groupby(col)[target]
    table = g.agg(
        n='count',
        p10=lambda s: s.quantile(0.10),
        p25=lambda s: s.quantile(0.25),
        p50='median',
        p75=lambda s: s.quantile(0.75),
        p90=lambda s: s.quantile(0.90),
    )
    return table[table['n'] >= min_count].sort_values('p50').round(1)

cat_features = ['region', 'seller_type', 'seller_user_type', 'room_type', 'rooms', 'is_new_building', 'is_apartments',
                'deal_conditions', 'building_type', 'renovation', 'parking', 'flat_type',
                'has_lift', 'mortgage_allowed', 'nearest_metro_walk',
                'demolished_in_renovation', 'is_penthouse', 'seller_is_owner']

for col in cat_features:
    print(col)
    display(target_by_category(dt, col))

В таблицах можем увидеть распределение кол-ва дней объявления на сайте по соответствующим квантилям

Посмотрим как различные факторы влияют на скорость продажи квартиры

In [ ]:
overlay_features = ['is_new_building', 'is_apartments', 'seller_type', 'building_type', 'deal_conditions']
clip = dt['days_on_market'].quantile(0.95)

for col in overlay_features:
    counts = dt[dt[col].ne('unknown')][col].value_counts()
    cats = counts[counts >= 500].index
    plt.figure()
    for val in cats:
        dt.loc[dt[col] == val, 'days_on_market'].plot(kind='kde', label=str(val))
    plt.title(f'Распределение days_on_market по {col}')
    plt.xlabel('days_on_market')
    plt.ylabel('Плотность')
    plt.xlim(0, clip)
    plt.legend()
    plt.show()

Посмотрим, где должна находиться квартира, чтобы ее быстрее можно было продать

In [ ]:
def lowest_median_bar(frame, col, target='days_on_market', top=10, min_count=100):
    sub = frame[frame[col].ne('unknown')]
    stats = sub.groupby(col)[target].agg(['median', 'count'])
    stats = stats[stats['count'] >= min_count].sort_values('median').head(top)
    plt.figure()
    plt.bar(stats.index, stats['median'])
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Топ-{top} {col}: наименьшая медиана days_on_market')
    plt.xlabel(col)
    plt.ylabel('Медиана days_on_market')
    plt.tight_layout()
    plt.show()

for col in ['district', 'municipality', 'nearest_metro']:
    lowest_median_bar(dt, col)

### Распределение таргета (price_per_m2) по категориальным признакам

In [ ]:
def target_by_category(frame, col, target='price_per_m2', min_count=200):
    g = frame.groupby(col)[target]
    table = g.agg(
        n='count',
        p10=lambda s: s.quantile(0.10),
        p25=lambda s: s.quantile(0.25),
        p50='median',
        p75=lambda s: s.quantile(0.75),
        p90=lambda s: s.quantile(0.90),
    )
    return table[table['n'] >= min_count].sort_values('p50').round(1)

cat_features = ['region', 'seller_type', 'seller_user_type', 'room_type', 'rooms', 'is_new_building', 'is_apartments',
                'deal_conditions', 'building_type', 'renovation', 'parking', 'flat_type',
                'has_lift', 'mortgage_allowed', 'nearest_metro_walk',
                'demolished_in_renovation', 'is_penthouse', 'seller_is_owner']

for col in cat_features:
    print(col)
    display(target_by_category(dt, col))

В таблицах можем увидеть распределение цены за кв. метр по соответствующим квантилям

Посмотрим как различные факторы влияют на цену квартиры за кв. метр

In [ ]:
overlay_features = ['is_new_building', 'is_apartments', 'seller_type', 'building_type', 'deal_conditions']
clip = dt['price_per_m2'].quantile(0.95)

for col in overlay_features:
    counts = dt[dt[col].ne('unknown')][col].value_counts()
    cats = counts[counts >= 500].index
    plt.figure()
    for val in cats:
        dt.loc[dt[col] == val, 'price_per_m2'].plot(kind='kde', label=str(val))
    plt.title(f'Распределение price_per_m2 по {col}')
    plt.xlabel('price_per_m2')
    plt.ylabel('Плотность')
    plt.xlim(0, clip)
    plt.legend()
    plt.show()

Посмотрим, где должна находиться квартира, чтобы ее можно было дороже продать

In [ ]:
def lowest_median_bar(frame, col, target='price_per_m2', top=10, min_count=100):
    sub = frame[frame[col].ne('unknown')]
    stats = sub.groupby(col)[target].agg(['median', 'count'])
    stats = stats[stats['count'] >= min_count].sort_values('median').head(top)
    plt.figure()
    plt.bar(stats.index, stats['median'])
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Топ-{top} {col}: наименьшая медиана price_per_m2')
    plt.xlabel(col)
    plt.ylabel('Медиана price_per_m2')
    plt.tight_layout()
    plt.show()

for col in ['district', 'municipality', 'nearest_metro']:
    lowest_median_bar(dt, col)

In [ ]:
new_cat = ['seller_user_type', 'room_type', 'demolished_in_renovation', 'is_penthouse', 'seller_is_owner']

fig, axes = plt.subplots(len(new_cat), 2, figsize=(16, 4 * len(new_cat)))
for row, col in enumerate(new_cat):
    sub = dt[dt[col].ne('unknown')] if dt[col].dtype == 'str' else dt
    order = sub[col].value_counts().index
    sns.boxplot(data=sub, x=col, y='days_on_market', order=order, ax=axes[row, 0], showfliers=False)
    axes[row, 0].set_title(f'days_on_market по {col}')
    axes[row, 0].tick_params(axis='x', rotation=30)
    sns.boxplot(data=sub, x=col, y='price_per_m2', order=order, ax=axes[row, 1], showfliers=False)
    axes[row, 1].set_title(f'price_per_m2 по {col}')
    axes[row, 1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### Анализ важности признаков

In [ ]:
interval_cols = [
    'days_on_market', 'price_last', 'price_per_m2', 'total_area', 'living_area',
    'kitchen_area', 'ceiling_height', 'floor', 'total_floors', 'building_age',
    'price_drop_pct', 'price_range_pct', 'dist_to_center', 'n_metro', 'nearest_metro_time', 'total_lifts',
    'floor_ratio', 'living_to_total', 'kitchen_to_total', 'area_per_room',
    'ppm2_to_district', 'ppm2_to_municipality',
    'bath_separate', 'bath_combined', 'balcony_count', 'loggia_count',
]
phik_cols = interval_cols + [
    'region', 'rooms', 'renovation', 'building_type', 'window_view',
    'parking', 'seller_type', 'seller_user_type', 'room_type', 'deal_conditions', 'flat_type', 'is_apartments', 'is_new_building',
    'phone_protected', 'had_discount', 'is_ready', 'event_closed',
    'is_first_floor', 'is_last_floor', 'is_studio', 'has_lift', 'mortgage_allowed', 'nearest_metro_walk',
    'demolished_in_renovation', 'is_penthouse', 'seller_is_owner',
]

phik_df = dt[phik_cols].copy()
phik_df[interval_cols] = phik_df[interval_cols].astype('float64')
for col in phik_cols:
    if str(phik_df[col].dtype) in ('boolean', 'bool', 'Int64'):
        phik_df[col] = phik_df[col].astype('int64').astype('object')

phik_matrix = phik_df.phik_matrix(interval_cols=interval_cols)

plot_correlation_matrix(
    phik_matrix.values,
    x_labels=phik_matrix.columns,
    y_labels=phik_matrix.index,
    vmin=0, vmax=1, color_map='Greens',
    title='Матрица корреляции признаков (phik)',
    fontsize_factor=1,
    figsize=(25, 25)
)
plt.tight_layout()
plt.show()

### Бейзлайн и анализ важности признаков

In [ ]:
leak_cols = ['days_on_market', 'first_seen', 'event_closed', 'price_points',
             'views_total', 'views_today', 'photos_count', 'had_discount', 'price_drop_pct', 'status']

y = dt['days_on_market'].astype(float)
X = dt.drop(columns=[c for c in leak_cols if c in dt.columns])

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')

X.shape, len(cat_cols)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=93,
    subsample=1,
    colsample_bytree=1,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train, categorical_feature=cat_cols)

pred = model.predict(X_test)
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
print('R2:', round(r2, 4))
print('MAE (дней):', round(mae, 1))

In [ ]:
lim = max(y_test.max(), pred.max())

plt.figure(figsize=(7, 7))
plt.scatter(y_test, pred, alpha=0.2, s=8)
plt.plot([0, lim], [0, lim], color='red')
plt.xlabel('Факт (days_on_market)')
plt.ylabel('Прогноз (days_on_market)')
plt.title(f'Pred vs Fact, R2 = {round(r2, 3)}')
plt.show()

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values().tail(20)

plt.figure(figsize=(8, 7))
plt.barh(importance.index, importance.values)
plt.xlabel('Важность (split count)')
plt.title('Топ-20 признаков LGBM')
plt.tight_layout()
plt.show()

### Финальный экспорт датасета

In [ ]:
final = dt.copy()
bool_cols = final.select_dtypes(include='bool').columns
final[bool_cols] = final[bool_cols].astype('int8')
final.to_csv('./notebooks/data/data_final.csv', index=False)
final.shape